## Preprocess the dataset

In [1]:
# Import libraries
from datasets import load_dataset
from transformers import AutoProcessor, AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import wave
import numpy as np
import pandas as pd
import os
import re
import string

/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.empty_cache()

In [3]:
# Retrieve .txt files
def list_files_in_directory(directory):
    file_list = []
    for filename in os.listdir(directory):
        # Only pick up files with .txt extensions (transcript)
        if filename.endswith(".txt"):
            file_list.append(filename.replace(".txt", ""))
    return file_list

# Create the dataframe
def get_reference_df(directory, audio_txt_file):
    txt_file_path = os.path.join(directory, audio_txt_file + ".txt")
    columns = ["start_time", "end_time", "reference"]
    # Read the text file into a DataFrame
    df = pd.read_csv(txt_file_path, sep="\t", header=None, names=columns, quoting=3)

    # Add file name
    df.insert(0, 'file_name', pd.Series([audio_txt_file] * len(df)))

    # Remove quotation marks
    df['reference'] = df['reference'].apply(lambda x : x.replace('"',""))
    
    return df

# Trim audio files
def trim_wav_by_timestamps(directory, wav_file_name, reference_df):
    # Create the output directory if it doesn't exist
    output_dir = "data/sub/"
    os.makedirs(output_dir, exist_ok=True)
    wav_file = os.path.join(directory, wav_file_name + ".wav") # get into data file
    
    # Load the WAV file
    audio = AudioSegment.from_wav(wav_file)
    
    def trim_segments(row):
        start_ms = float(row['start_time']) * 1000  # Convert start time to milliseconds
        end_ms = float(row['end_time']) * 1000      # Convert end time to milliseconds
        trimmed_segment = audio[start_ms:end_ms]
    
        return trimmed_segment
    
    # Iterate over timestamps and trim the audio
    for i, row in reference_df.iterrows():
        trimmed_segment = trim_segments(row)
        output_file = os.path.join(output_dir, wav_file_name + "_" f"trimmed_segment_{i+1}.wav")
        trimmed_segment.export(output_file, format="wav")
        reference_df.at[i, 'trimmed_segment_path'] = output_file
    
    return reference_df

# Helper function to remove punctuations from original subtitles
def strip_punctuation(text):
    # Create a translation table that maps each punctuation character to None
    translator = str.maketrans('', '', string.punctuation)
    # Use the translation table to remove all punctuation from the text
    return text.translate(translator)

# Filter english text and append it to dataframe
def filter_subs_by_lang(reference_df):
    # Helper function that is applied across the rows to filter english text only
    
    def filter_english_only(text):
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Join the English words into a single string
        english_text = ' '.join(english_words)
        return english_text
    
    def filter_thai_only(text):
        # Remove punctuation from text
        text = strip_punctuation(text)
        # Tokenize the string, split by spaces
        list_of_words = text.split()
        # Define a regex pattern to match English words
        english_pattern = re.compile(r'\b[A-Za-z]+\b')
        # Find all English words in the text
        english_words = english_pattern.findall(text)
        # Find all Thai words
        thai_words = [word for word in list_of_words if word not in english_words]
        # Concatenate the Thai words into a string
        thai_text = ' '.join(thai_words)
        # Return thai string
        return thai_text

    reference_df['eng_reference'] = reference_df['reference'].apply(filter_english_only)
    reference_df['thai_reference'] = reference_df['reference'].apply(filter_thai_only)

    return reference_df

def get_combined_audio_table(directory, file_names):
    combined_df = pd.DataFrame()
    for file_name in file_names:
        # Reads the transcript dataframe which has the start_time, end_time of each transcript
        reference_df = get_reference_df(directory, file_name)

        # Split subtitles by language
        reference_df = filter_subs_by_lang(reference_df)
        # Remove 'reference' column
        reference_df = reference_df.drop('reference', axis=1)

        # Uncomment to trim all the .wav file according to the subtitles start_time and end_time
        ## reference_df = trim_wav_by_timestamps(directory, file_name, reference_df)
        
        # Append the processed DataFrame to the combined DataFrame
        combined_df = pd.concat([combined_df, reference_df], ignore_index=True)

        # Comment out this section if not required
        combined_df = combined_df.drop(['file_name', 'start_time', 'end_time'], axis=1)
    
    return combined_df

directory = os.path.join(os.getcwd(), "data/train/")
file_names = list_files_in_directory(directory)
combined_df = get_combined_audio_table(directory, file_names)

pd.options.display.max_colwidth = 100
combined_df

,eng_reference,thai_reference
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน
3,a month of giving out bonuses,เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ
4,And our company is,และบริษัทของเราเนี่ยเป็น
...,...,...
2408,Please like share and subscribe,ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย
2409,Chollada Channel Chollada Channel,กับ ค่ะ
2410,Bye,บาย
2411,Please like share and subscribe,อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์


## Import SEALION Model

In [4]:
generation_kwargs = {
    "do_sample": False,  # set to true if temperature is not 0
    "temperature": None,
    "max_new_tokens": 256,
    "top_k": 50,
    "top_p": 0.7,
    "repetition_penalty": 1.2,
}

In [5]:
MODEL_NAME = "aisingapore/sea-lion-7b-instruct"

# Import tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "aisingapore/sea-lion-7b-instruct", 
    trust_remote_code=True
)

generation_kwargs["eos_token_id"] = tokenizer.eos_token_id

In [6]:
# Remove unneeded kwargs
if generation_kwargs["do_sample"] == False:
    generation_kwargs.pop("temperature")
    generation_kwargs.pop("top_k")
    generation_kwargs.pop("top_p")

In [7]:
# Import model
model = AutoModelForCausalLM.from_pretrained(
    "aisingapore/sea-lion-7b-instruct",
    trust_remote_code=True,
    device_map = 'auto'
)

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.50s/it]


The code below generates the output in the form of tokens.

In [48]:
# Example Use Case

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"
prompt = """กินอาหาร"""
full_prompt = prompt_template.format(human_prompt=prompt)

tokens = tokenizer(full_prompt, return_tensors="pt")

# Output in the form of tokens
output = model.generate(
    tokens["input_ids"],
    attention_mask = tokens["attention_mask"],
    **generation_kwargs,
    #return_dict_in_generate=True, 
    #output_logits=True
)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/utils.py:1659: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


### USER:
Translate the following to English. กินอาหาร

### RESPONSE:
To eat food


In [35]:
output

tensor([[ 71721,  66561, 249853,      4,  88172,    311,   2057,    332,   3839,
         249835, 108918,      4,      4,  29864, 229553, 249853,      4,  12953,
              1]])

The code below generates the output in the form of logits.
Each tensor in the output tuple represents one token.

In [50]:
# Example Use Case

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"
prompt = """กินอาหาร"""
full_prompt = prompt_template.format(human_prompt=prompt)

tokens = tokenizer(full_prompt, return_tensors="pt")

output = model.generate(
    tokens["input_ids"],
    attention_mask = tokens["attention_mask"],
    **generation_kwargs,
    return_dict_in_generate=True, 
    output_logits=True
)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/utils.py:1659: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


In [106]:
tuple_logits = output.logits

In [107]:
stacked_logits = torch.stack(tuple_logits)
stacked_logits[:-1][:][:]

tensor([[[-3.1207, 12.3748, -3.1105,  ..., -3.4588, -0.5081,  0.4274]],

        [[-4.1495, 13.9526, -4.1255,  ..., -3.9904, -2.2551, -0.7501]],

        [[-5.6498, 24.0050, -5.6453,  ..., -4.0959, -1.6021, -1.5405]]])

In [108]:
prob_logits = stacked_logits[:-1][:][:].softmax(dim=-1)
prob_logits[0]

tensor([[1.3860e-11, 7.4360e-05, 1.4001e-11,  ..., 9.8831e-12, 1.8896e-10,
         4.8157e-10]])

torch.Size([4, 1, 256000])


tensor([[249845]])

In [1]:
# Test
test_df = combined_df.head(3)

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"

def generate_translation(reference_df):

    def translate_thai(prompt):
        full_prompt = prompt_template.format(human_prompt=prompt)
        # Retrieve tokens
        tokens = tokenizer(full_prompt, return_tensors="pt")
        # Generate output
        output = model.generate(
            tokens["input_ids"],
            attention_mask = tokens["attention_mask"],
            **generation_kwargs,
        )
        # Decode output embeddings
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        print(decoded_output)
        # Trim to obtain output from formatted response
        translation = decoded_output.partition("RESPONSE:\n")[2]
        
        return translation

    reference_df['translation'] = reference_df['thai_reference'].apply(translate_thai)
    return reference_df

test_df = generate_translation(test_df)
test_df

NameError: name 'combined_df' is not defined

In [16]:
# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"

# Generate tokens and generate output
def generate_translation(reference_df):

    def translate_thai(prompt):
        full_prompt = prompt_template.format(human_prompt=prompt)
        # Retrieve tokens
        tokens = tokenizer(full_prompt, return_tensors="pt")
        # Generate output
        output = model.generate(
            tokens["input_ids"],
            attention_mask = tokens["attention_mask"],
            **generation_kwargs,
        )
        # Decode output embeddings
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        # Trim to obtain output from formatted response
        translation = decoded_output.partition("RESPONSE:\n")[2]

        return translation

    reference_df['translation'] = reference_df['thai_reference'].apply(translate_thai)
    return reference_df

combined_df = generate_translation(combined_df)
combined_df

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.pyenv/versions/3.11.7/lib/python3.11/site-packages/transformers/generation/utils.py:1659: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


,eng_reference,thai_reference,translation
0,So today I will treat them to,เดี๋ยววันเนี่ยจะพาไปเลี้ยง,Let's go out for dinner tonight.
1,spicy Tom Yum hot pot,ต้มยำหม้อไฟแซ่บๆ,Let's make some hot and spicy soup!
2,Hello everyone This is,สวัสดีครับทุกคน เดือนนี้เป็นเดือน,"Hello everyone, it is June"
3,a month of giving out bonuses,เมษายนเป็นเดือนแห่งการแจกโบนัสนะครับ,"April is a month of bonus distribution, right?"
4,And our company is,และบริษัทของเราเนี่ยเป็น,And our company is
...,...,...,...
2408,Please like share and subscribe,ดูแล้วชอบ กดไลก์ แชร์ ซับสไครบ์ด้วย,"I like this video, please click Like and Share with your friends!"
2409,Chollada Channel Chollada Channel,กับ ค่ะ,ใช้คำว่า with แทนได้ไหมคะ?
2410,Bye,บาย,Good evening
2411,Please like share and subscribe,อย่าลืมกดไลก์ กดแชร์ หรือว่าซับสไครบ์,"If you are a student, please do not use this service for academic purposes such as submitting as..."


## Inspect SEA-LION Model

### Inspect TOKEN Output Format

In [9]:
# Import LLM
tokenizer = AutoTokenizer.from_pretrained(
    "aisingapore/sea-lion-7b-instruct", 
    trust_remote_code=True
)

prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"
prompt = "ฉันหิว"
full_prompt = prompt_template.format(human_prompt=prompt)

# Preprocess the input data (Convert audio to tensors)
tokens = tokenizer(full_prompt, return_tensors="pt")
print(tokens)

{'input_ids': tensor([[ 71721,  66561, 249853,      4,  88172,    311,   2057,    332,   3839,
         249835,  66012, 226626,      4,      4,  29864, 229553, 249853,      4]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [55]:
# Input must be a tensor
embeddings = model.transformer.wte(tokens['input_ids'])
embeddings
embeddings.size()

torch.Size([1, 18, 4096])

In [57]:
# Example Use Case

output = model.generate(
    inputs_embeds = embeddings,
    **generation_kwargs,
)
print(output)
print(tokenizer.decode(output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


tensor([[249842,    725,  18847, 249835,      1]], device='cuda:0')
I am hungry.


In [64]:
output[0]

tensor([249842,    725,  18847, 249835,      1], device='cuda:0')

In [65]:
tokens = tokenizer.decode(output[0])
print(tokens)

 I am hungry.<|endoftext|>


: 

### Inspect SEALION Model for Batch Inputs

In [8]:
test_eng_data = combined_df.loc[30:31, "eng_reference"]
test_eng_data = test_eng_data.tolist()
print(test_eng_data)

test_thai_data = combined_df.loc[30:31, "thai_reference"]
test_thai_data = test_thai_data.tolist()
print(test_thai_data)

['No more work Do you know what we ll have', 'No I don t Is it spicy and flavourful']
['ไม่ต้องทำงาน รู้ยังว่ากินไร', 'ไม่รู้แต่ว่ามันแซ่บไหมล่ะ']


In [9]:
# Example Use Case

# Prompt template
prompt_template = "### USER:\nTranslate the following to English. {human_prompt}\n\n### RESPONSE:\n"

def create_full_prompts(list_of_inputs):
    full_prompt_list = []
    # Iterate through list of inputs (thai text)
    for prompt in list_of_inputs:
        full_prompt = prompt_template.format(human_prompt = prompt)
        full_prompt_list.append(full_prompt)

    return full_prompt_list

full_prompt_list = create_full_prompts(test_thai_data)

# Get tokens of list of prompts
tokenizer.padding_side = "left"
tokens = tokenizer(full_prompt_list, return_tensors="pt", padding=True)

# Get embeddings of list of prompts
embeddings = model.transformer.wte(tokens['input_ids'])
print(embeddings.shape)

output = model.generate(
    inputs_embeds = embeddings,
    attention_mask = tokens["attention_mask"],
    **generation_kwargs,
    return_dict_in_generate=True, 
    output_logits=True
)

tuple_logits = output.logits

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


torch.Size([2, 23, 4096])


/home/dhuser/.cache/huggingface/modules/transformers_modules/aisingapore/sea-lion-7b-instruct/566afff3d12a72de5e2a0a627d233ad940fe7dcb/attention.py:124: UserWarning: Propagating key_padding_mask to the attention module and applying it within the attention module can cause unnecessary computation/memory usage. Consider integrating into attn_bias once and passing that to each attention module instead.
  warnings.warn(


In [10]:
tokens["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

In [11]:
output.sequences

tensor([[  5275, 249866, 249815,    675, 249832,    357,    810,    727,    332,
           4501, 249835,      1,      1],
        [ 18435,    364,    986, 249866, 249815,    810, 249832,    556,    399,
            370,  24549, 249835,      1]], device='cuda:0')

In [12]:
output.sequences.shape

torch.Size([2, 13])

In [13]:
# To obtain English translation, if logits is not required
tokens_without_special_tokens = tokenizer.batch_decode(output.sequences, skip_special_tokens=True)
print(tokens_without_special_tokens)

tokens_with_special_tokens = tokenizer.batch_decode(output.sequences, skip_special_tokens=False)
print(tokens_with_special_tokens)

["Don't work, I know what to eat.", "Maybe you don't know, but it is spicy."]
[" Don't work, I know what to eat.<|endoftext|><|endoftext|>", " Maybe you don't know, but it is spicy.<|endoftext|>"]


Note: Final output is also padded to have same length.

### Inspect Cross Entropy for Loss Function

In [10]:
## Obtain logits for translated output

# Convert tuple of tensors to a single tensor with 3D dimensions
stacked_logits = torch.stack(tuple_logits)

# Remove the last token, since it represents </end_of_text>
stacked_logits = stacked_logits[:-1][:][:]

# Convert logits to probabilities
prob_logits = stacked_logits.softmax(dim=-1)

# [max_length_of_generated_seq][batch_size][vocab_size]
print(prob_logits.shape)

prob_logits_transposed = prob_logits.transpose(0, 1)
# [batch_size][max_length_of_generated_seq][vocab_size]
print(prob_logits_transposed.shape)

torch.Size([12, 2, 256000])
torch.Size([2, 12, 256000])


In [11]:
## Obtain tokens for actual subtitles
tokenizer.padding_side = "right"
# Set padding token to <end_of_text> token, which has id 1
tokenizer.pad_token_id = 1

tokens_for_test = tokenizer(test_eng_data, return_tensors="pt", padding=True)
target_ids = tokens_for_test["input_ids"]
print(target_ids)

tensor([[  1813,    573,    675,   2061,    364,    810,    727,    427,  48061,
            479],
        [  1813,    357,    986,    302,   1634,    399,  24549,    336, 161289,
              1]])


In [12]:
def compare_generated_and_actual(prob_logits_transposed, target_ids):

    max_len_generated_int = prob_logits_transposed.size(1)

    max_len_original_int = target_ids.size(1)

    diff_in_len = max_len_generated_int - max_len_original_int

    return diff_in_len

In [13]:
def pad_actual(target_ids, diff_in_len):
    # Pad on right side of last dimension
    padding_template = (0, diff_in_len)
    # Pad actual tokens with <end_of_text>
    target_ids = torch.nn.functional.pad(target_ids, padding_template, "constant", 1)
    # Return modified actual sequence
    return target_ids

vocab_size = 256000
batch_size = 2

def pad_generated_seq(generated_logits, diff_in_len):
    # Create padding tensor that represents <end_of_text> with token id 1
    padding_logit = torch.zeros((batch_size, 1, vocab_size))
    padding_logit[0,0,1] = 1.0
    padding_logit = padding_logit.repeat(1, diff_in_len, 1)
    # Pad generated sequence along dimension 1 (length of sequence)
    padded_logit = torch.cat((generated_logits, padding_logit), dim=1)
    # Return padded logit
    return padded_logit

def padding_process(diff_in_len, target_ids, generated_logits):
    # Case 1: Generated sequence is longer than actual
    if (diff_in_len > 0):
        target_ids = pad_actual(target_ids, diff_in_len)
        return target_ids
    
    # Case 2: Generated sequence is shorter than actual
    elif (diff_in_len < 0):
        generated_logits = pad_generated_seq(generated_logits)
        
    # Case 3: Same maximum length
    else:
        print("Max length of generated = Max length of actual in the batch")

In [15]:
# Calculate loss 
import torch.nn.functional as F

diff_in_len = compare_generated_and_actual(prob_logits_transposed, target_ids)
target_ids = pad_actual(target_ids, diff_in_len)

prob_logits_transposed = prob_logits_transposed.to("cpu")
target_ids = target_ids.to("cpu")

loss = F.cross_entropy(
    prob_logits_transposed.reshape(-1, prob_logits_transposed.shape[-1]), target_ids.view(-1)
)
print(loss)

tensor(12.3629)


In [25]:
prob_logits

tensor([[[6.5276e-11, 8.6510e-05, 6.4579e-11,  ..., 2.9246e-10,
          3.0253e-09, 7.7853e-09],
         [5.4681e-10, 4.0890e-03, 5.4932e-10,  ..., 1.1605e-09,
          3.2917e-08, 1.9536e-08]],

        [[8.1905e-15, 1.4349e-06, 8.1538e-15,  ..., 4.3931e-13,
          1.0283e-12, 2.7974e-12],
         [4.1497e-15, 1.0687e-03, 4.1811e-15,  ..., 6.6684e-13,
          6.2684e-12, 1.6311e-11]],

        [[2.6892e-12, 1.8237e-07, 2.7067e-12,  ..., 1.6734e-12,
          4.4226e-12, 2.5847e-11],
         [4.6216e-15, 3.0842e-07, 4.6487e-15,  ..., 3.4286e-13,
          9.7290e-13, 2.2367e-12]],

        ...,

        [[6.2934e-16, 4.0747e-05, 6.3738e-16,  ..., 3.0519e-15,
          2.5109e-14, 5.0120e-14],
         [2.0709e-13, 2.0404e-06, 2.1029e-13,  ..., 9.3097e-13,
          2.0331e-12, 1.3517e-11]],

        [[3.5908e-15, 1.4449e-02, 3.5574e-15,  ..., 5.1006e-14,
          5.6567e-13, 1.2666e-12],
         [1.9516e-11, 4.4121e-05, 1.9877e-11,  ..., 3.6781e-11,
          1.1450e-10, 1